In [ ]:
from pathlib import Path


def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "pretrain" / "train_mae2d_geom.py").is_file():
            return p
    raise FileNotFoundError("Open this notebook inside the AR-SSL4M repo.")


REPO = find_repo(Path.cwd())
NPY_DIR = REPO / "EWS" / "data" / "npy"
LIST_DIR = REPO / "EWS" / "data" / "pretrain" / "patch_lists"
LIST_FILE = LIST_DIR / "all_patches.txt"
CKPT_DIR_GEOM = REPO / "EWS" / "data" / "pretrain" / "checkpoints_geom"

if not NPY_DIR.is_dir():
    raise FileNotFoundError(f"Prepare this directory first: {NPY_DIR}")

LIST_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR_GEOM.mkdir(parents=True, exist_ok=True)


In [ ]:
count_1 = 0
count_2 = 0

paths = sorted(NPY_DIR.glob("*.npy"))
with open(LIST_FILE, "w", encoding="utf-8") as f:
    for p in paths:
        f.write(str(p.resolve()) + "\n")
        count_1 += 1
        count_2 += 1
        if count_2 >= 100:
            print(count_1)
            count_2 = 0

print("list_path:", LIST_FILE)
print("num_samples:", len(paths))


In [ ]:
import sys

PRE = REPO / "EWS" / "pretrain"
sys.path.insert(0, str(PRE))

from train_mae2d_geom import run_training

USE_GEOM_BIAS = True  # set False to ablate (standard attention stack, no distance bias)

run_training(
    list_path=str(LIST_FILE.resolve()),
    ckpt_dir=str(CKPT_DIR_GEOM.resolve()),
    img_size=350,
    patch_size=5,
    epochs=1,
    batch_size=1,
    num_workers=2,
    val_every=500,
    mask_ratio=0.5,
    use_geom_bias=USE_GEOM_BIAS,
)

_ckpt = CKPT_DIR_GEOM.resolve()
print("Geom checkpoints saved under:", _ckpt)
for p in sorted(_ckpt.glob("mae2d_geom*.pt")):
    print(" ", p.name, p.stat().st_size // (1024 * 1024), "MB")
